In [ ]:
print("hello")

In [5]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

# 1. Load the dataset from scikit-learn
data = load_breast_cancer()

# 2. Convert to a pandas DataFrame
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target  # 1 = Benign, 0 = Malignant

# 3. Split into Train (80%) and Test (20%) sets
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['target'])

# 4. Save the full dataset and the test dataset
test_df.to_csv('test_data.csv', index=False)
train_df.to_csv('train_data.csv', index=False)

print("Dataset prepared successfully!")
print(f"Train set shape: {train_df.shape}")
print(f"Test set shape: {test_df.shape}")

Dataset prepared successfully!
Train set shape: (455, 31)
Test set shape: (114, 31)


In [6]:
import pandas as pd
import numpy as np
import pickle
import os
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score, 
    recall_score, f1_score, matthews_corrcoef
)

# Ensure the 'model' directory exists
os.makedirs('model', exist_ok=True)

# 1. Load the data
train_df = pd.read_csv('train_data.csv')
test_df = pd.read_csv('test_data.csv')

# Split features and targets
X_train = train_df.drop(columns=['target'])
y_train = train_df['target']
X_test = test_df.drop(columns=['target'])
y_test = test_df['target']

# 2. Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save the scaler for use in the Streamlit app
with open('model/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# 3. Define the 5 models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'kNN': KNeighborsClassifier(),
    'Naive Bayes': GaussianNB(),
    'Random Forest (Ensemble)': RandomForestClassifier(random_state=42)
}

# 4. Dictionary to store metrics for the comparison table
results = []

# 5. Train, evaluate, and save each model
for name, model in models.items():
    # Train the model
    # Use scaled data for all models to keep the pipeline consistent
    model.fit(X_train_scaled, y_train)
    
    # Predict labels and probabilities
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]  # Probability for AUC
    
    # Calculate the 6 evaluation metrics
    accuracy = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    
    # Append results
    results.append({
        'ML Model Name': name,
        'Accuracy': round(accuracy, 4),
        'AUC': round(auc, 4),
        'Precision': round(precision, 4),
        'Recall': round(recall, 4),
        'F1': round(f1, 4),
        'MCC': round(mcc, 4)
    })
    
    # Save the trained model file inside the 'model' directory
    model_filename = f"model/{name.lower().replace(' ', '_').replace('(', '').replace(')', '')}.pkl"
    with open(model_filename, 'wb') as f:
        pickle.dump(model, f)
    print(f"Trained and saved: {name}")

# 6. Display the comparison table
results_df = pd.DataFrame(results)
print("\n=== Model Comparison Table ===")
print(results_df.to_string(index=False))

Trained and saved: Logistic Regression
Trained and saved: Decision Tree
Trained and saved: kNN
Trained and saved: Naive Bayes
Trained and saved: Random Forest (Ensemble)

=== Model Comparison Table ===
           ML Model Name  Accuracy    AUC  Precision  Recall     F1    MCC
     Logistic Regression    0.9825 0.9954     0.9861  0.9861 0.9861 0.9623
           Decision Tree    0.9123 0.9157     0.9559  0.9028 0.9286 0.8174
                     kNN    0.9561 0.9788     0.9589  0.9722 0.9655 0.9054
             Naive Bayes    0.9298 0.9868     0.9444  0.9444 0.9444 0.8492
Random Forest (Ensemble)    0.9561 0.9939     0.9589  0.9722 0.9655 0.9054
